In [1]:
import os, json, time
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── 설정 ─────────────────────────────────────────────────
load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = "gpt-4o-mini"

df = pd.read_csv('./sfscore_df.csv', encoding='utf-8-sig')
np.random.seed(42)
df_sample = df.sample(n=50).reset_index(drop=True)
print(f"샘플: {len(df_sample)}건 | 5인 에이전트 × 2조건 = {len(df_sample)*5*2}회 API 호출")

# ── 에이전트 정의 ─────────────────────────────────────────
agents_with_ref = [
    {"role": "수석 언더라이터", "focus": "인수 지침 준수 및 역선택 방지",
     "ref": "FSC(2021) 보험업권 AI 거버넌스 가이드라인에 따라"},
    {"role": "임상 전문의", "focus": "의학적 인과관계 및 질병 위험도",
     "ref": "Singhal et al.(2023) 의료 LLM 임상 성능 연구 및 INTERHEART 코호트(Yusuf et al., 2004)에 따라"},
    {"role": "보험 계리사", "focus": "통계적 변동성 및 손해율 산정",
     "ref": "Klugman et al.(2012) 손해보험 계리학 이론에 따라"},
    {"role": "예방의학 전문가", "focus": "생활습관 장기 영향 및 수정 가능 위험인자",
     "ref": "Khaw et al.(2008) 코호트 연구 및 WHO(2022) 만성질환 예방 가이드라인에 따라"},
    {"role": "준법 감시인", "focus": "판정 공정성 및 소비자 보호",
     "ref": "Jobin et al.(2019) AI 윤리 원칙 및 NAIC(2023) 보험 알고리즘 가이드라인에 따라"},
]

agents_without_ref = [
    {"role": "수석 언더라이터", "focus": "인수 지침 준수 및 역선택 방지", "ref": ""},
    {"role": "임상 전문의", "focus": "의학적 인과관계 및 질병 위험도", "ref": ""},
    {"role": "보험 계리사", "focus": "통계적 변동성 및 손해율 산정", "ref": ""},
    {"role": "예방의학 전문가", "focus": "생활습관 장기 영향 및 수정 가능 위험인자", "ref": ""},
    {"role": "준법 감시인", "focus": "판정 공정성 및 소비자 보호", "ref": ""},
]

GRADE_MAP = {'Standard': 0, 'Loading': 1, 'Decline': 2}

def make_system(agent):
    ref_str = f"{agent['ref']} " if agent['ref'] else ""
    return (
        f"당신은 보험 언더라이팅 {agent['role']}입니다. "
        f"{ref_str}{agent['focus']}에 집중하여 심사하십시오.\n"
        "반드시 JSON 형식으로만 응답하십시오: "
        '{"grade": "Standard/Loading/Decline", "score": 0.0~1.0, "reason": "간략한 근거"}'
    )

def make_user(sogyeon):
    return f"다음 보험 소견서를 심사하여 판정하십시오.\n\n{sogyeon[:2000]}"

def judge_one(args):
    idx, row, agent, condition = args
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": make_system(agent)},
                {"role": "user", "content": make_user(row['합성_소견서'])}
            ],
            temperature=0,
            max_tokens=256,
        )
        raw = response.choices[0].message.content.strip()
        raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        result = json.loads(raw)
        return {
            'idx': idx, 'condition': condition,
            'role': agent['role'],
            'grade': result.get('grade', 'Unknown'),
            'score': float(result.get('score', 0.5)),
            'error': None
        }
    except Exception as e:
        return {
            'idx': idx, 'condition': condition,
            'role': agent['role'],
            'grade': 'Unknown', 'score': 0.5, 'error': str(e)
        }

# ── 병렬 실행 ─────────────────────────────────────────────
tasks = []
for i, row in df_sample.iterrows():
    for agent in agents_with_ref:
        tasks.append((i, row, agent, 'with_ref'))
    for agent in agents_without_ref:
        tasks.append((i, row, agent, 'without_ref'))

results = []
done = 0
errors = 0

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(judge_one, t): t for t in tasks}
    for f in as_completed(futures):
        r = f.result()
        results.append(r)
        done += 1
        if r['error']:
            errors += 1
        if done % 50 == 0:
            print(f"진행: {done}/{len(tasks)}건 (오류: {errors}건)", flush=True)

print(f"\n완료. 성공: {done-errors}건 / 오류: {errors}건")
df_res = pd.DataFrame(results)

# ── 조건별 최빈값(최종 판정) 산출 ────────────────────────
def get_final_grade(grades):
    valid = [g for g in grades if g in GRADE_MAP]
    if not valid:
        return 'Unknown'
    return max(set(valid), key=valid.count)

summary = []
for idx in range(len(df_sample)):
    for cond in ['with_ref', 'without_ref']:
        sub = df_res[(df_res['idx'] == idx) & (df_res['condition'] == cond)]
        grades = sub['grade'].tolist()
        final = get_final_grade(grades)
        avg_score = sub['score'].mean()
        summary.append({
            'idx': idx, 'condition': cond,
            'final_grade': final,
            'avg_score': avg_score
        })

df_sum = pd.DataFrame(summary)

# ── 결과 비교 ────────────────────────────────────────────
print("\n" + "=" * 65)
print("학술 근거 유무에 따른 판정 분포 비교 (N=50건)")
print("=" * 65)

for cond, label in [('with_ref', '학술 근거 있음'), ('without_ref', '학술 근거 없음')]:
    sub = df_sum[df_sum['condition'] == cond]
    dist = sub['final_grade'].value_counts()
    print(f"\n▶ {label}")
    for g in ['Standard', 'Loading', 'Decline']:
        cnt = dist.get(g, 0)
        print(f"   {g}: {cnt}건 ({cnt/len(sub)*100:.1f}%)")
    print(f"   평균 위험점수: {sub['avg_score'].mean():.4f}")

# ── 일치율 분석 ───────────────────────────────────────────
df_pivot = df_sum.pivot(index='idx', columns='condition', values='final_grade')
agree = (df_pivot['with_ref'] == df_pivot['without_ref']).sum()
total = len(df_pivot)
print(f"\n▶ 판정 일치율: {agree}/{total}건 ({agree/total*100:.1f}%)")
print(f"▶ 판정 불일치: {total-agree}건 ({(total-agree)/total*100:.1f}%)")

# ── 위험점수 차이 통계 검정 ───────────────────────────────
w = df_sum[df_sum['condition'] == 'with_ref']['avg_score'].values
wo = df_sum[df_sum['condition'] == 'without_ref']['avg_score'].values
t_stat, p_val = stats.ttest_rel(w, wo)
print(f"\n▶ 위험점수 대응표본 T-test")
print(f"   With ref 평균: {w.mean():.4f}")
print(f"   Without ref 평균: {wo.mean():.4f}")
print(f"   t={t_stat:.4f}, p={'< .001' if p_val < 0.001 else round(p_val, 3)}")
print(f"   통계적 유의차: {'있음' if p_val < 0.05 else '없음'}")

df_sum.to_csv('academic_ref_effect_result.csv', index=False, encoding='utf-8-sig')
print("\n결과 저장: academic_ref_effect_result.csv")

샘플: 50건 | 5인 에이전트 × 2조건 = 500회 API 호출
진행: 50/500건 (오류: 0건)
진행: 100/500건 (오류: 0건)
진행: 150/500건 (오류: 0건)
진행: 200/500건 (오류: 0건)
진행: 250/500건 (오류: 0건)
진행: 300/500건 (오류: 0건)
진행: 350/500건 (오류: 0건)
진행: 400/500건 (오류: 0건)
진행: 450/500건 (오류: 0건)
진행: 500/500건 (오류: 0건)

완료. 성공: 500건 / 오류: 0건

학술 근거 유무에 따른 판정 분포 비교 (N=50건)

▶ 학술 근거 있음
   Standard: 14건 (28.0%)
   Loading: 32건 (64.0%)
   Decline: 4건 (8.0%)
   평균 위험점수: 0.6654

▶ 학술 근거 없음
   Standard: 13건 (26.0%)
   Loading: 36건 (72.0%)
   Decline: 1건 (2.0%)
   평균 위험점수: 0.6868

▶ 판정 일치율: 46/50건 (92.0%)
▶ 판정 불일치: 4건 (8.0%)

▶ 위험점수 대응표본 T-test
   With ref 평균: 0.6654
   Without ref 평균: 0.6868
   t=-2.9023, p=0.006
   통계적 유의차: 있음

결과 저장: academic_ref_effect_result.csv
